# Table 4.5 — MDR Weight Perturbation (rebuilt from the authoritative pipeline)

This notebook **reuses your own `xai_19_06_2025.py`** — the exact script that produced `spillnet_xai_comprehensive_results.json` (2026-02-23). It reloads `best_model.keras`, reselects the **same 20 samples** (via `np.random.seed(42)`), regenerates the five explanation maps, and captures the three MDR sub-components (EP, SC, BS) per technique per sample. It then re-weights those components across your five weight scenarios.

**Why this reconciles:** the sub-component maths and the 0.40/0.30/0.30 weighting are taken verbatim from `marine_domain_relevance()`, so the **Original** column must reproduce your published MDR means (Grad-CAM 0.347, SHAP 0.299, LIME 0.268, IG 0.252, LRP 0.232). Cell 6 is a hard reconciliation gate — if the Original column does not match, **stop and do not use the output**.

**Honesty note:** this is a *weight-perturbation sensitivity* analysis on the same 20-sample set. Grad-CAM (Sobel-based EP) is the definition your published results use — run this, not the boundary-attention (`FIXED_EP`) variant, or it will not reconcile.

In [ ]:
# 1) Dependencies
!pip install -q opencv-python-headless scikit-image shap lime scipy 2>/dev/null
import numpy as np, tensorflow as tf
print('TF', tf.__version__, '| NumPy', np.__version__)

In [ ]:
# 2) Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3) Bring in YOUR script and import it (reuses your exact model loader, generators, and MDR maths)
import os, sys, shutil

# EDIT this if your script lives elsewhere in Drive:
SCRIPT_SRC = '/content/drive/MyDrive/xai_19_06_2025.py'

if not os.path.exists(SCRIPT_SRC):
    # fall back to manual upload
    from google.colab import files
    print('Script not found at', SCRIPT_SRC, '- upload xai_19_06_2025.py now:')
    up = files.upload()
    SCRIPT_SRC = '/content/' + list(up.keys())[0]

shutil.copy(SCRIPT_SRC, '/content/xai_module.py')
sys.path.insert(0, '/content')
import xai_module as xai   # runs a harmless self-test print on import
print('\nImported. Model path in CONFIG:', xai.CONFIG['model_path'])
print('num_test_samples:', xai.CONFIG['num_test_samples'])

In [ ]:
# 3b) Fix CONFIG paths for the umahatokula@gmail.com Drive layout
# The dataset moved from DrAdeyinka/Oil_Spill_Detection_Dataset to My Drive > PhD > dataset > Oil_Spill_Detection_Dataset
xai.CONFIG['data_dir'] = '/content/drive/MyDrive/PhD/dataset/Oil_Spill_Detection_Dataset'

# The model checkpoint under this account may live somewhere other than SpillNet_QuickRestart_Results/models/
# List every .keras/.h5 file in this Drive so you can pick the one that matches your published results:
import glob
found_ckpts = glob.glob('/content/drive/MyDrive/**/*.keras', recursive=True) + glob.glob('/content/drive/MyDrive/**/*.h5', recursive=True)
print('Found checkpoints:')
for p in found_ckpts:
    print(' -', p)

# Once you see the right file above (the one that matches your published MDR numbers), set it explicitly:
# xai.CONFIG['model_path'] = '/content/drive/MyDrive/<correct/path>/best_model.keras'

print('
Updated data_dir:', xai.CONFIG['data_dir'])
print('Current model_path (edit above if wrong):', xai.CONFIG['model_path'])

In [ ]:
# 4) MDR sub-components — copied VERBATIM from marine_domain_relevance() so re-weighting is exact.
#    Returns the raw (pre-final-clip) EP, SC, BS. mdr_at() re-applies any weight set + the same [0,1] clip.
from scipy import ndimage

def mdr_components(explanation, ground_truth):
    edges = ndimage.sobel(explanation)
    oil = ground_truth == 1
    ep = float(np.mean(edges[oil])) if oil.sum() > 0 else 0.0
    if oil.sum() > 0:
        eio = explanation[oil]
        sc = float(1.0 / (1.0 + np.std(eio) / (np.mean(eio) + 1e-8)))
    else:
        sc = 0.0
    sea = ground_truth == 0
    if sea.sum() > 0 and oil.sum() > 0:
        supp = 1.0 - (np.mean(explanation[sea]) / (np.mean(explanation[oil]) + 1e-8))
        bs = float(max(0.0, min(1.0, supp)))
    else:
        bs = 0.0
    return ep, sc, bs

def mdr_at(ep, sc, bs, w):
    return float(max(0.0, min(1.0, w[0]*ep + w[1]*sc + w[2]*bs)))

# sanity: at original weights this equals xai.marine_domain_relevance() by construction
print('components fn ready')

In [ ]:
# 5) Load model + the SAME 20 sample pairs (seed 42 inside load_actual_dataset)
model = xai.load_trained_spillnet()
if isinstance(model, (list, tuple)):
    model = model[0]
pairs = xai.load_actual_dataset()
print(f'\nLoaded model and {len(pairs)} sample pairs')

In [ ]:
# 6) Regenerate explanations and capture EP/SC/BS per technique per sample
techniques = {
    'Grad-CAM':            lambda m, img: xai.grad_cam_spillnet(m, img, class_idx=1),
    'LIME':                lambda m, img: xai.simple_lime(m, img),
    'SHAP':                lambda m, img: xai.simple_shap(m, img),
    'IntegratedGradients': lambda m, img: xai.integrated_gradients(m, img, steps=20),
    'LRP':                 lambda m, img: xai.layer_wise_relevance_propagation(m, img),
}

comp = {t: {'EP': [], 'SC': [], 'BS': []} for t in techniques}

for i, (img_path, mask_path) in enumerate(pairs):
    img, mask, err = xai.preprocess_data_pair(img_path, mask_path)
    if err:
        print(f'  [{i+1}] skip: {err}')
        continue
    for t, fn in techniques.items():
        try:
            expl = fn(model, img)
            ep, sc, bs = mdr_components(expl, mask)
        except Exception as e:
            print(f'  [{i+1}] {t} failed: {e}')
            ep, sc, bs = 0.0, 0.0, 0.0
        comp[t]['EP'].append(ep); comp[t]['SC'].append(sc); comp[t]['BS'].append(bs)
    print(f'  processed {i+1}/{len(pairs)}')

n = len(comp['Grad-CAM']['EP'])
print(f'\nCaptured components for {n} samples x {len(techniques)} techniques')

In [ ]:
# 7) RECONCILIATION GATE — Original (0.40/0.30/0.30) must reproduce published MDR means
AUTH = {'Grad-CAM': 0.3467, 'LIME': 0.2678, 'SHAP': 0.2989,
        'IntegratedGradients': 0.2525, 'LRP': 0.2324}  # from spillnet_xai_comprehensive_results.json
W_ORIG = (0.40, 0.30, 0.30)
TOL = 0.03

print(f"{'Technique':22s} {'recomputed':>11s} {'published':>10s} {'delta':>8s}")
ok = True
for t in techniques:
    vals = [mdr_at(comp[t]['EP'][k], comp[t]['SC'][k], comp[t]['BS'][k], W_ORIG) for k in range(n)]
    m = float(np.mean(vals)); d = m - AUTH[t]
    flag = '' if abs(d) <= TOL else '  <-- MISMATCH'
    if abs(d) > TOL: ok = False
    print(f'{t:22s} {m:11.4f} {AUTH[t]:10.4f} {d:+8.4f}{flag}')

print()
if ok:
    print('RECONCILED (within tol). Deterministic techniques should be near-exact;')
    print('small SHAP/LIME deltas are expected from perturbation randomness.')
else:
    print('DID NOT RECONCILE. Do NOT use the table below. Likely causes:')
    print(' - wrong EP definition (must be Sobel, not boundary-attention FIXED_EP)')
    print(' - different dataset folder / glob order -> different 20 samples')
    print(' - IG steps not 20, or a different model file')

In [ ]:
# 8) Build Table 4.5 across the five weight scenarios
scenarios = {
    'Original (0.40/0.30/0.30)': (0.40, 0.30, 0.30),
    'A (0.35/0.35/0.30)':        (0.35, 0.35, 0.30),
    'B (0.45/0.25/0.30)':        (0.45, 0.25, 0.30),
    'C (0.50/0.25/0.25)':        (0.50, 0.25, 0.25),
    'D (0.30/0.35/0.35)':        (0.30, 0.35, 0.35),
}
order = ['Grad-CAM', 'IntegratedGradients', 'SHAP', 'LRP', 'LIME']
disp  = {'Grad-CAM':'Grad-CAM','IntegratedGradients':'IG','SHAP':'SHAP','LRP':'LRP','LIME':'LIME'}

def scenario_means(w):
    return {t: float(np.mean([mdr_at(comp[t]['EP'][k], comp[t]['SC'][k], comp[t]['BS'][k], w)
                              for k in range(n)])) for t in order}

rows = {}
for name, w in scenarios.items():
    means = scenario_means(w)
    ranking = sorted(means, key=means.get, reverse=True)   # rank 1 = highest MDR
    rank = {t: ranking.index(t) + 1 for t in order}
    rows[name] = {t: (rank[t], means[t]) for t in order}

# print as a clean table
hdr = 'Scenario'.ljust(26) + ''.join(disp[t].center(16) for t in order)
print(hdr); print('-'*len(hdr))
for name in scenarios:
    line = name.ljust(26)
    for t in order:
        rk, mv = rows[name][t]
        line += f'{rk} ({mv:.3f})'.center(16)
    print(line)

In [ ]:
# 9) Save provenance (per-sample components + scenario table) so the table is fully traceable
import json, datetime
out = {
    'generated': datetime.datetime.now().isoformat(),
    'source_model': xai.CONFIG['model_path'],
    'n_samples': n,
    'ep_definition': 'sobel (original, matches published MDR)',
    'weight_scenarios': {k: list(v) for k, v in scenarios.items()},
    'per_sample_components': comp,
    'scenario_table': {name: {disp[t]: {'rank': rows[name][t][0], 'mdr': round(rows[name][t][1], 3)}
                              for t in order} for name in scenarios},
}
save_path = '/content/drive/MyDrive/mdr_weight_perturbation_results.json'
json.dump(out, open(save_path, 'w'), indent=2)
print('Saved provenance to', save_path)
print('\nPaste the table from cell 8 back into the chat and I will format Table 4.5 + rewrite 4.4.4.1/4.4.4.2.')